In [0]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

catalog_name = 'ecommerce'

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

In [0]:
df_silver = df_bronze.withColumn('brand_name', F.trim(F.col('brand_name')))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn('brand_code', F.regexp_replace(F.col('brand_code'), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
# Replace repeating category codes

# Anomalies dictionary
anomalies = {
    "GROCERY" : "GRCY",
    "BOOKS" : "BKS",
    "TOYS" : "TOY",
}

df_silver = df_silver.replace(anomalies, subset="category_code")

df_silver.select("category_code").distinct().show()

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

**Category**

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")

df_bronze.show(10)

In [0]:
# Show duplicates

df_duplicates = df_bronze.groupBy("category_code").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
# Remove duplicates

df_silver = df_bronze.dropDuplicates(['category_code'])
display(df_silver)

In [0]:
# Convert the category code to uppercase

df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))
display(df_silver)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_category)

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

In [0]:
display(df_bronze.limit(5))

**Check `weight_grams` (contains 'g')**

In [0]:
# Check weight_grams column
df_bronze.select("weight_grams").show(5, truncate=False)

In [0]:
# replace 'g' with ''
df_silver = df_bronze.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
df_silver.select("weight_grams").show(5, truncate=False)

**Check `length_cm` (comma instead of dot)**

In [0]:
df_silver.select("length_cm").show(3)

In [0]:
# replace , with .
df_silver = df_silver.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType())
)
df_silver.select("length_cm").show(3)

`category_code` and `brand_code` are in lower case. we need to make it all upper case

In [0]:
df_silver.select("category_code", "brand_code").show(2)

In [0]:
# convert category_code and brand_code to upper case
df_silver = df_silver.withColumn(
    "category_code",
    F.upper(F.col("category_code"))
).withColumn(
    "brand_code",
    F.upper(F.col("brand_code"))
)
df_silver.select("category_code", "brand_code").show(2)

Spelling mistakes in `material` column

In [0]:
df_silver.select("material").distinct().show()

In [0]:
# Fix spelling mistakes
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)
df_silver.select("material").distinct().show()    

Negative values in `rating_count`

In [0]:
df_silver.filter(F.col('rating_count')<0).select("rating_count").show(3)

In [0]:
# Convert negative rating_count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

In [0]:
# Check final cleaned data

df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_dim_products)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")